In [1]:
!pip install -q google-genai sentence-transformers chromadb \
    langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [2]:
import os

from google.colab import userdata, files
from google import genai

from sentence_transformers import SentenceTransformer
import chromadb

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [4]:
api_key = os.environ.get("GOOGLE_API_KEY") or os.environ.get("G_A_K")

if not api_key:
    try:
        api_key = userdata.get("Gemini_API_Key")
    except Exception:
        pass

if not api_key:
    raise ValueError(
        "Gemini API key not found. "
        "Add 'Gemini_API_key' to Google Colab Secrets."
    )

os.environ["GOOGLE_API_KEY"] = api_key

# Create Gemini client
client = genai.Client(api_key=api_key)

In [7]:
print("Please upload one or more PDF files:")

uploaded = files.upload()

pdf_texts = []

for filename in uploaded.keys():

    if filename.lower().endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page_num, page in enumerate(reader.pages):

            page_text = page.extract_text()

            if page_text:
                text += f"\n--- Page {page_num + 1} ---\n" + page_text

            pdf_texts.append(text)

            print(f"loaded '{filename}' ({len(reader.pages)} pages).")

if not pdf_texts:
    raise ValueError(
        "No valid PDF files were uploaded. "
        "Please upload at least one PDF file."
    )

    full_pdf_content = "\n\n".join(pdf_texts)

Please upload one or more PDF files:


Saving Basic LLM Chatbot Using Gemini_85.pdf to Basic LLM Chatbot Using Gemini_85 (2).pdf
loaded 'Basic LLM Chatbot Using Gemini_85 (2).pdf' (2 pages).
loaded 'Basic LLM Chatbot Using Gemini_85 (2).pdf' (2 pages).


In [12]:
full_pdf_content = ""

for pdf in pdf_texts:

    if isinstance(pdf, dict):
        full_pdf_content += pdf["text"] + "\n\n"

    else:
        full_pdf_content += str(pdf) + "\n\n"

if not full_pdf_content.strip():
    raise ValueError("No text was extracted from the PDF.")

print("PDF content extracted successfully.")
print("Total characters:", len(full_pdf_content))

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_text(full_pdf_content)

print(
    f"Extracted and split document into "
    f"{len(chunks)} text chunks."
)

PDF content extracted successfully.
Total characters: 2395
Extracted and split document into 7 text chunks.


In [13]:
print("Loading embedding model and building vector index...")

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="pdf_rag_collection"
)

chunk_embeddings = embedder.encode(chunks).tolist()

chunk_ids = [
    f"doc_chunk_{i}"
    for i in range(len(chunks))
]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings,
    ids=chunk_ids
)

print("Vector index created successfully!")
print(f"Total chunks stored: {collection.count()}")

Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector index created successfully!
Total chunks stored: 7


In [26]:
# =========================================================
# ASK PDF USING GEMINI 3.6 FLASH
# =========================================================

def ask_pdf(query: str):

    # -----------------------------------------------------
    # RETRIEVE RELEVANT PDF CHUNKS
    # -----------------------------------------------------

    context_passages = retrieve_pdf_context(
        query,
        top_k=3
    )

    # -----------------------------------------------------
    # COMBINE PDF CONTEXT
    # -----------------------------------------------------

    context_str = "\n\n".join(context_passages)

    # -----------------------------------------------------
    # CREATE PROMPT
    # -----------------------------------------------------

    prompt = f"""
You are an intelligent PDF document analysis assistant.

Answer the question using ONLY the information
contained in the PDF context below.

If the answer is not available in the context, say:

"I cannot find the answer in the provided document."

Do not make up information.

================ PDF CONTEXT ================

{context_str}

================ QUESTION ================

{query}

================ ANSWER ================
"""

    # -----------------------------------------------------
    # GEMINI 3.6 FLASH - INTERACTIONS API
    # -----------------------------------------------------

    response = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    # -----------------------------------------------------
    # GET GEMINI ANSWER
    # -----------------------------------------------------

    answer = response.output_text

    # Return answer + retrieved context
    return answer, context_passages

In [27]:
# =========================================================
# PDF CHATBOT
# =========================================================

print("=" * 60)
print("          PDF CHATBOT READY!")
print("  Type your question below.")
print("  Type 'exit' to quit.")
print("=" * 60)

while True:

    user_query = input("\nAsk a question about your PDF: ")

    if user_query.lower().strip() in ["exit", "quit", "q"]:
        print("\nExiting PDF Chatbot. Goodbye!")
        break

    if not user_query.strip():
        print("Please enter a question.")
        continue

    try:

        answer, context = ask_pdf(user_query)

        print("\n--- RETRIEVED PDF SNIPPETS ---")

        for i, snippet in enumerate(context, 1):
            print(f"\n[{i}] {snippet[:150]}...")

        print("\n--- GEMINI RESPONSE ---")
        print(answer)

        print("\n" + "=" * 60)

    except Exception as e:

        print("\n❌ Error occurred:")
        print(e)

        print("=" * 60)

          PDF CHATBOT READY!
  Type your question below.
  Type 'exit' to quit.

Ask a question about your PDF: What is this PDF about?

--- RETRIEVED PDF SNIPPETS ---

[1] --- Page 2 ---
Name:- Vibha Kamble   Roll No:- 85 
 
 
Step 4 : Initialize the Gemini Client 
 
Step 5 : Define the Prompt 
 
Step 6 : Generate the Re...

[2] --- Page 1 ---
Name:- Vibha Kamble   Roll No:- 85 
 
Basic LLM Chatbot Using Gemini 
1. Title :- 
Basic LLM Chatbot Using Google Gemini API 
2. Aim :-...

[3] --- Page 1 ---
Name:- Vibha Kamble   Roll No:- 85 
 
Basic LLM Chatbot Using Gemini 
1. Title :- 
Basic LLM Chatbot Using Google Gemini API 
2. Aim :-...

--- GEMINI RESPONSE ---
This PDF is an assignment report by Vibha Kamble about creating a **Basic LLM Chatbot Using the Google Gemini API** in Google Colab. 

Key details provided in the document include:
* **Aim:** To create a simple Large Language Model (LLM) application using the Google Gemini API in Google Colab to generate answers for user questio